# ADM01 · M08 전출 증가와 조회 급증을 나누어 시험하기 (학습자용)

이 노트북은 가상 조직 4곳과 행정지원 직무 6종의 자료를 다루는 **미션8(M08) 전용** 실습 파일입니다. **현재 소속된 인원과 당장 근무할 수 있는 인원을 구분하고, 앞으로 13주 동안 필요한 보충 인원을 근거와 함께 설명하는 것**이 목표입니다.

준비 셀을 먼저 실행하면 가상 자료와 공통 도구를 만듭니다. 별도의 `ADM01.md` 파일을 업로드할 필요가 없습니다. 공통 코드에는 자료 생성기·검증기·파서·예측 함수·권한 검사·HTML 저장 도구가 있습니다.

미션 내 다섯 단계(`STEP 01`~`STEP 05`)를 순서대로 실행합니다. 결과 HTML은 왼쪽 `STEP 01`~`05` 메뉴로 단계별 계산을 보여 주며, 같은 폴더에 전체 CSV와 실행 기록도 저장합니다. 모델 정확도, 속도, 브라우저 측정은 실제 수행한 범위만 보고합니다.

- 실제 부대·장병·군번·운영자료는 사용하지 않습니다.
- 가용 인원이 0명이면 1인당 초과근무 평균은 미측정입니다.
- 제안과 승인, 실제 집행 확인은 다른 상태입니다.
- 이 노트북의 모의 역할은 실제 인사시스템의 인증 체계를 대신하지 않습니다.

학습자용 파일(`ADM01_m08_student.ipynb`)입니다. 미션8의 5개 미작성 함수는 `NotImplementedError`로 중단됩니다. 설명을 읽고 함수 본문을 완성하세요.

### 가상 직무 코드

직무 코드는 J01 행정지원, J02 기록관리, J03 시설지원, J04 교육운영 지원, J05 급식지원, J06 물품관리로 읽는다. 실제 편제나 보직을 나타내는 코드가 아니라 사례를 구분하는 가상 이름이다.

In [ ]:
from pathlib import Path
import sys, json, time, importlib.util, platform
PROJECT_HOME = Path.cwd() / 'ADM01_work'
PROJECT_HOME.mkdir(exist_ok=True)
COMMON_CODE = '"""ADM01 교육용 실행 도구. 개인자료·실제 부대자료·네트워크 연결을 사용하지 않는다."""\nfrom pathlib import Path\nfrom functools import lru_cache\nimport copy, csv, hashlib, html, io, json, math, queue, re, threading, time, sys, traceback\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import Ridge\nVERSION = \'ADM01-2.0\'\nUNITS=[f\'B{i:02}\' for i in range(1,5)]\nROLES=[f\'J{i:02}\' for i in range(1,7)]\nFIELDS=[\'week\',\'unit\',\'role\',\'authorized\',\'current\',\'leave\',\'sick\',\'training\',\'detached\',\'overtime_h\',\'night_ratio\',\'shift_h\',\'inflow\',\'outflow\',\'source_id\']\nKEY=[\'week\',\'unit\',\'role\']\nABSENT=[\'leave\',\'sick\',\'training\',\'detached\']\nHORIZON=13\nORIGINS=[52,65,78,91]\n\ndef native(x):\n    if isinstance(x,(np.integer,)): return int(x)\n    if isinstance(x,(np.floating,)): return float(x) if np.isfinite(x) else None\n    if isinstance(x,np.ndarray): return [native(v) for v in x.tolist()]\n    if isinstance(x,dict): return {str(k):native(v) for k,v in x.items()}\n    if isinstance(x,(list,tuple)): return [native(v) for v in x]\n    if isinstance(x,(Path,)): return str(x)\n    if isinstance(x,pd.DataFrame): return json.loads(x.to_json(orient=\'records\',force_ascii=False))\n    if x is pd.NA: return None\n    return x\n\ndef dump(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(native(value),ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\')\n    return path\n\ndef fingerprint(df):\n    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:16]\n\ndef make_weekly(seed=7,shock=False):\n    """실제 유입·유출 인원을 기록한 뒤 현재원을 갱신합니다."""\n    rng=np.random.default_rng(seed); rows=[]; labels=[]\n    for ui,u in enumerate(UNITS):\n        for ji,r in enumerate(ROLES):\n            auth=24+ji*3+ui*2; current=auth-int(rng.integers(0,5))\n            for w in range(1,105):\n                surge=bool(shock and ui<2 and w>=84)\n                seasonal=1.8 if w%52 in range(19,26) else 1.0\n                out=min(current,int(rng.poisson(.35*seasonal+(1.1 if surge else 0))))\n                incoming=int(rng.poisson(.25))+(int(rng.integers(2,5)) if w%13==0 else 0)\n                # 예정된 유입과 실제 유입을 구분합니다. 상한을 적용한 차이는 실제 유입량에도 반영합니다.\n                inflow=min(incoming,max(auth-current+out,0))\n                current=current+inflow-out\n                absent=rng.multinomial(current,[.055,.025,.045,.015,.86])[:4]\n                avail=current-int(absent.sum())\n                workload=auth*40*(.96+.025*ji)\n                noise=float(rng.normal(0,1.2))\n                ot=max(0,workload/avail-40+noise) if avail>0 else np.nan\n                night=float(np.clip(rng.normal(.18+.035*ji,.06),0,.65))\n                # 정답 라벨은 점수식을 복사하지 않고 별도의 교육용 사례 규칙으로 생성합니다.\n                y=int((ot>=13 and night>=.2) or (auth-current>=7 and ot>=8) or (absent.sum()/max(current,1)>=.24))\n                rows.append([w,u,r,auth,current,*map(int,absent),round(ot,2),round(night,3),[8,12,8,8,12,8][ji],inflow,out,f\'synth-{seed}-w{w:03}\'])\n                labels.append([w,u,r,y])\n    return pd.DataFrame(rows,columns=FIELDS),pd.DataFrame(labels,columns=KEY+[\'load_label\'])\n\ndef derive(df):\n    x=df.copy();x[\'absent\']=x[ABSENT].sum(axis=1)\n    x[\'vacancy\']=(x.authorized-x.current).clip(lower=0)\n    x[\'surplus\']=(x.current-x.authorized).clip(lower=0)\n    x[\'available\']=x.current-x.absent\n    x[\'vacancy_rate\']=x.vacancy/x.authorized\n    x[\'workload_observed\']=x.available.gt(0)&x.overtime_h.notna() if \'overtime_h\' in x else x.available.gt(0)\n    return x\n\ndef default_schema():\n    descriptions={\n      \'week\':(\'주\',\'주말을 기준으로 한 주차\'),\'unit\':(\'코드\',\'가상 조직\'),\'role\':(\'코드\',\'가상 행정지원 직무\'),\n      \'authorized\':(\'명\',\'편제에 정한 인원\'),\'current\':(\'명\',\'기준시점 현재원\'),\'leave\':(\'명\',\'휴가 부재 인원\'),\n      \'sick\':(\'명\',\'진단정보 없는 환자 부재 집계\'),\'training\':(\'명\',\'교육 부재 인원\'),\'detached\':(\'명\',\'현재원에 포함된 파견 부재\'),\n      \'overtime_h\':(\'시간/인/주\',\'해당 주 가용 인원 1인당 초과근무 평균\'),\'night_ratio\':(\'비율\',\'전체 근무시간 중 야간 비율\'),\n      \'shift_h\':(\'시간\',\'한 번에 연속으로 근무하는 시간\'),\'inflow\':(\'명/주\',\'지난 주말 이후 실제 유입\'),\'outflow\':(\'명/주\',\'지난 주말 이후 실제 유출\'),\n      \'source_id\':(\'코드\',\'가상 생성 배치 참조\')}\n    properties={}\n    for k in FIELDS:\n        typ=\'string\' if k in [\'unit\',\'role\',\'source_id\'] else \'number\' if k in [\'overtime_h\',\'night_ratio\',\'shift_h\'] else \'integer\'\n        rule={\'type\':typ,\'unit\':descriptions[k][0],\'description\':descriptions[k][1],\'source\':\'교육용 생성기 \'+VERSION}\n        if typ!=\'string\':rule[\'minimum\']=1 if k in [\'week\',\'authorized\',\'shift_h\'] else 0\n        else:rule[\'minLength\']=1\n        if k==\'week\':rule[\'maximum\']=104\n        if k==\'night_ratio\':rule[\'maximum\']=1\n        if k==\'overtime_h\':rule[\'nullable_when\']=\'현재원과 중복 없는 부재 합계가 같아 가용 인원이 0명일 때만 미측정으로 허용\'\n        if k==\'unit\':rule[\'enum\']=UNITS\n        if k==\'role\':rule[\'enum\']=ROLES\n        properties[k]=rule\n    return {\'$schema\':\'https://json-schema.org/draft/2020-12/schema\',\'type\':\'object\',\'required\':FIELDS,\'properties\':properties,\'additionalProperties\':False}\n\ndef validate(df,schema=None,check_stock=True):\n    """원본 행 위치를 보존하고 오류행과 사유를 반환합니다. 입력은 수정하지 않습니다."""\n    schema=schema or default_schema(); errs={i:[] for i in range(len(df))}\n    missing=[k for k in FIELDS if k not in df]\n    for pos,rec in enumerate(df.to_dict(\'records\')):\n        extra=set(rec)-set(FIELDS)\n        if extra:errs[pos].append(\'UNKNOWN_FIELDS:\'+\',\'.join(sorted(extra)))\n        for k,rule in schema[\'properties\'].items():\n            if k not in rec:errs[pos].append(k+\':MISSING_COLUMN\');continue\n            v=rec[k]\n            if pd.isna(v):\n                zero_available=(k==\'overtime_h\' and all(isinstance(rec.get(x),(int,float,np.number)) for x in [\'current\']+ABSENT) and rec[\'current\']==sum(rec[x] for x in ABSENT))\n                if not zero_available:errs[pos].append(k+\':MISSING_VALUE\')\n                continue\n            numeric=isinstance(v,(int,float,np.number)) and not isinstance(v,(bool,np.bool_))\n            valid_type=isinstance(v,str) if rule[\'type\']==\'string\' else numeric and (rule[\'type\']!=\'integer\' or float(v).is_integer())\n            if not valid_type:errs[pos].append(k+\':TYPE\');continue\n            if numeric and not np.isfinite(v):errs[pos].append(k+\':NONFINITE\');continue\n            if \'enum\' in rule and v not in rule[\'enum\']:errs[pos].append(k+\':ENUM\')\n            if isinstance(v,str) and not v.strip():errs[pos].append(k+\':EMPTY\')\n            if \'minimum\' in rule and v<rule[\'minimum\']:errs[pos].append(k+\':MIN\')\n            if \'maximum\' in rule and v>rule[\'maximum\']:errs[pos].append(k+\':MAX\')\n        if not errs[pos] and sum(rec[k] for k in ABSENT)>rec[\'current\']:errs[pos].append(\'ABSENT_EXCEEDS_CURRENT\')\n    if not missing:\n        dup=df.duplicated(KEY,keep=False)\n        for pos in np.flatnonzero(dup):errs[int(pos)].append(\'DUPLICATE_KEY\')\n        if check_stock:\n            work=df.copy();work[\'_pos\']=np.arange(len(df))\n            for _,g in work.sort_values(KEY).groupby([\'unit\',\'role\']):\n                previous=None\n                for rec in g.to_dict(\'records\'):\n                    pos=rec[\'_pos\']\n                    if errs[pos]:previous=None;continue\n                    if previous is not None and rec[\'week\']==previous[\'week\']+1:\n                        if rec[\'current\']!=previous[\'current\']+rec[\'inflow\']-rec[\'outflow\']:errs[pos].append(\'STOCK_MISMATCH\')\n                    previous=rec\n    bad=np.array([bool(errs[i]) for i in range(len(df))]);clean=df.iloc[np.flatnonzero(~bad)].copy()\n    quarantine=df.iloc[np.flatnonzero(bad)].copy();quarantine[\'row_position\']=np.flatnonzero(bad)\n    quarantine[\'errors\']=[\';\'.join(errs[i]) for i in np.flatnonzero(bad)]\n    return clean,quarantine\n\ndef schema_cases(df):\n    cases={};base=df.head(1).copy()\n    for name,col,value in [(\'TYPE\',\'authorized\',\'서른\'),(\'ENUM\',\'role\',\'J99\'),(\'NEGATIVE\',\'current\',-1),(\'RATIO\',\'night_ratio\',1.2),(\'MISSING_VALUE\',\'night_ratio\',np.nan),(\'ABSENT\',\'leave\',999)]:\n        x=base.copy().astype(object);x.loc[x.index[0],col]=value;cases[name]=x\n    cases[\'MISSING_COLUMN\']=base.drop(columns=\'night_ratio\')\n    cases[\'DUPLICATE_KEY\']=pd.concat([base,base],ignore_index=True)\n    return cases\n\nCLOCK=re.compile(r\'(\\d{1,2}):(\\d{2})\\s*~\\s*(익일\\s*)?(\\d{1,2}):(\\d{2})\')\nKOCLOCK=re.compile(r\'(오전|오후)\\s*(\\d{1,2})시\\s*~\\s*(익일\\s*)?(오전|오후)\\s*(\\d{1,2})시\')\nDURATION=re.compile(r\'(\\d+(?:\\.\\d+)?)\\s*시간(?:\\s*(\\d+)\\s*분)?\')\nHOURS=re.compile(r\'(\\d+(?:\\.\\d+)?)\\s*h\\b\')\nREASONS={\'휴가\':[\'휴가\',\'연가\'],\'환자\':[\'환자\',\'진료\',\'병가\'],\'교육\':[\'교육\',\'입교\'],\'파견\':[\'파견\'],\'전출\':[\'전출\',\'전역\']}\n\ndef parse_hours(text):\n    m=CLOCK.search(text)\n    if m:\n        a,am,nxt,b,bm=m.groups();a,am,b,bm=map(int,[a,am,b,bm])\n        if a>23 or b>23 or am>59 or bm>59:raise ValueError(\'시각 범위 오류\')\n        diff=b*60+bm-a*60-am\n        if nxt:diff+=1440\n        elif diff<0:diff+=1440 # 문제에서 정한 한 번의 자정 통과 규칙\n        elif diff==0:raise ValueError(\'같은 시각의 기간 미확인\')\n        return diff/60\n    m=KOCLOCK.search(text)\n    if m:\n        ap,a,nxt,bp,b=m.groups();a=int(a);b=int(b)\n        if not 1<=a<=12 or not 1<=b<=12:raise ValueError(\'오전 오후 시각 범위 오류\')\n        diff=(b%12+(12 if bp==\'오후\' else 0))-(a%12+(12 if ap==\'오후\' else 0))\n        if nxt:diff+=24\n        elif diff<0:diff+=24\n        elif diff==0:raise ValueError(\'같은 시각의 기간 미확인\')\n        return float(diff)\n    for regex in [DURATION,HOURS]:\n        m=regex.search(text)\n        if m:\n            minute=int(m[2]) if regex is DURATION and m[2] else 0\n            if minute>=60:raise ValueError(\'분 범위 오류\')\n            return float(m[1])+minute/60\n    raise ValueError(\'시간 표현이 완전하지 않음\')\n\ndef parse_line(text):\n    unit=re.search(r\'\\bB0[1-4]\\b\',text);role=re.search(r\'\\bJ0[1-6]\\b\',text);context=re.search(r\'평시|야간|훈련주간\',text)\n    if not (unit and role and context):raise ValueError(\'조직 직무 상황 누락\')\n    tail=text.split(\'사유=\',1)[-1] if \'사유=\' in text else \'\'\n    matched=[k for k,words in REASONS.items() if any(w in tail for w in words)]\n    reason=matched[0] if len(matched)==1 else \'미확인\'\n    return dict(unit=unit[0],role=role[0],context=context[0],hours=parse_hours(text),reason=reason)\n\ndef make_logs(seed=12):\n    rng=np.random.default_rng(seed);raw=[];gold=[]\n    # 일정한 사례 수를 고정한다. 내용과 정답은 서로 다른 표에 저장한다.\n    broken=set(range(0,6000,60));anomalous=set(range(1,6000,80))\n    for i in range(6000):\n        u=UNITS[i%4];r=ROLES[i%6];ctx=[\'평시\',\'야간\',\'훈련주간\'][i%3];reason=list(REASONS)[i%5]\n        h=round(float(np.clip(rng.normal({\'평시\':9,\'야간\':11,\'훈련주간\':15}[ctx],1.1),4,20))*2)/2\n        if i in anomalous:h=99.0 if (i//80)%2 else .5\n        form=i%5\n        if form==0:t=f\'{int(h)}시간 {int(round((h%1)*60))}분\'\n        elif form==1:t=f\'{h:g}h\'\n        elif form==2:\n            a=22;b=int(a*60+h*60);t=f\'22:00~\'+(\'익일 \' if b>=1440 else \'\')+f\'{b//60%24:02}:{b%60:02}\'\n            if h>=24:t=f\'총 {h:g}시간\'\n        elif form==3:\n            # 오전·오후 시각은 정수 시간으로 생성하며 정답도 같은 기간으로 저장합니다.\n            h=float(int(h));end=14+int(h)\n            if h>=24:t=f\'총 {h:g}시간\'\n            else:t=\'오후 2시~\'+(\'익일 \' if end>=24 else \'\')+(\'오후 \' if end%24>=12 else \'오전 \')+f\'{end%12 or 12}시\'\n        else:t=f\'총 {h:g}시간\'\n        original=f\'{u} {r} [{ctx}] 근무 {t} 사유={reason}\'\n        parseable=i not in broken\n        if not parseable:original=f\'{u} {r} [{ctx}] 근무 [잘린 기록]\'\n        rid=f\'L{i:05}\';raw.append([rid,original,None])\n        gold.append([rid,parseable,h if parseable else None,reason if parseable else None,i in anomalous,False])\n    for i in range(60):\n        rid=f\'R{i:05}\';raw.append([rid,raw[i][1],raw[i][0]])\n        gold.append([rid,*gold[i][1:5],True])\n    return pd.DataFrame(raw,columns=[\'line_id\',\'text\',\'duplicate_of\']),pd.DataFrame(gold,columns=[\'line_id\',\'parseable\',\'truth_hours\',\'truth_reason\',\'is_anomaly\',\'is_duplicate\'])\n\ndef preprocess(raw):\n    rows=[]\n    for item in raw.to_dict(\'records\'):\n        rec=dict(item);rec[\'status\']=\'DUPLICATE\' if pd.notna(item.get(\'duplicate_of\')) else \'PARSED\'\n        try:rec.update(parse_line(item[\'text\']));rec[\'error\']=None\n        except ValueError as exc:\n            rec.update(hours=np.nan,reason=\'미확인\',unit=None,role=None,context=None,error=str(exc))\n            if rec[\'status\']!=\'DUPLICATE\':rec[\'status\']=\'PARSE_FAILED\'\n        rows.append(rec)\n    return pd.DataFrame(rows)\n\ndef binary_metrics(truth,pred):\n    y=np.asarray(truth,dtype=bool);p=np.asarray(pred,dtype=bool)\n    tp=int((y&p).sum());fp=int((~y&p).sum());fn=int((y&~p).sum());tn=int((~y&~p).sum())\n    div=lambda a,b:float(a/b) if b else None\n    return dict(n=len(y),tp=tp,fp=fp,fn=fn,tn=tn,accuracy=div(tp+tn,len(y)),precision=div(tp,tp+fp),recall=div(tp,tp+fn),f1=div(2*tp,2*tp+fp+fn),fpr=div(fp,fp+tn),fdr=div(fp,tp+fp))\n\ndef features_load(df):\n    d=derive(df)\n    return np.column_stack([np.clip(d.vacancy_rate/.25,0,1),np.clip(d.overtime_h/20,0,1),np.clip(d.night_ratio/.4,0,1)])\n\ndef load_score(df,weights=(.3,.6,.1)):\n    if len(weights)!=3 or min(weights)<0 or not np.isclose(sum(weights),1):raise ValueError(\'가중치 3개의 합은 1이어야 함\')\n    return features_load(df)@np.asarray(weights)*100\n\ndef get_cell(df,u,r,origin):\n    g=df[(df.unit==u)&(df.role==r)&(df.week<=origin)].sort_values(\'week\').copy()\n    if g.empty:raise ValueError(\'해당 시점에 자료가 없음\')\n    return g\n\ndef season(week):return 1.0 if week%52 in range(19,26) else 0.0\n\ndef ridge_features(v,i,o,next_week,v2=True):\n    x=[v[-1],v[-1]-v[-2],v[-1]-v[-4]]\n    if v2:x += [np.mean(o[-8:])-np.mean(i[-8:]),float(next_week%13==0),season(next_week)]\n    return x\n\ndef fit_ridge(df,origin,v2=True):\n    X=[];y=[]\n    for _,g in df[df.week<=origin].sort_values(\'week\').groupby([\'unit\',\'role\']):\n        v=(g.authorized-g.current).clip(lower=0).to_numpy()/g.authorized.to_numpy();i=g.inflow.to_numpy()/g.authorized.to_numpy();o=g.outflow.to_numpy()/g.authorized.to_numpy()\n        weeks=g.week.to_numpy()\n        for t in range(12,len(g)):\n            if weeks[t]-weeks[t-4]!=4:continue\n            X.append(ridge_features(v[:t],i[:t],o[:t],int(weeks[t]),v2));y.append(v[t]-v[t-1])\n    if not X:raise ValueError(\'학습용 연속 주차 부족\')\n    return Ridge(alpha=1).fit(X,y)\n\ndef ridge_predict(model,g,origin,v2=True):\n    a=float(g.authorized.iloc[-1]);v=list(((g.authorized-g.current).clip(lower=0)/g.authorized).to_numpy());i=list(g.inflow/a);o=list(g.outflow/a);out=[]\n    for h in range(1,14):\n        d=float(model.predict([ridge_features(v,i,o,origin+h,v2)])[0]);v.append(float(np.clip(v[-1]+d,0,1)));out.append(v[-1]*a)\n        i.append(float(np.mean(i[-8:])));o.append(float(np.mean(o[-8:])))\n    return np.array(out)\n\ndef mc_paths(g,origin,n_paths=300,seed=0,lam=0.0,extra_out=0.0):\n    """예측시점까지의 실적만으로 13주 경로를 만듭니다. 확정된 일정과 구분합니다."""\n    if not 0<=lam<=1 or extra_out<0:raise ValueError(\'예측 설정 범위 오류\')\n    g=g[g.week<=origin].sort_values(\'week\');rng=np.random.default_rng(seed)\n    if g.empty:raise ValueError(\'예측시점 이전 자료 없음\')\n    a=int(g.authorized.iloc[-1]);current=np.full(n_paths,int(g.current.iloc[-1]),dtype=int)\n    seasonal=g.week.map(season).to_numpy().astype(bool)\n    out_mean=float(g.outflow.mean());base=float(g.loc[~seasonal,\'outflow\'].mean()) if (~seasonal).any() else out_mean\n    seasonal_out=float(g.loc[seasonal,\'outflow\'].mean()) if seasonal.any() else base*1.8\n    recent=float(g.tail(8).outflow.mean());ordinary=g.week%13!=0\n    incoming=float(g.loc[ordinary,\'inflow\'].mean()) if ordinary.any() else float(g.inflow.mean())\n    batch=max(0,float(g.loc[~ordinary,\'inflow\'].mean())-incoming) if (~ordinary).any() else 0\n    arr=np.zeros((n_paths,13)); flows=[]\n    for h in range(1,14):\n        mu=(1-lam)*(seasonal_out if season(origin+h) else base)+lam*recent+extra_out\n        out=np.minimum(current,rng.poisson(max(mu,0),n_paths))\n        proposed=rng.poisson(max(incoming,0),n_paths)+(rng.poisson(batch,n_paths) if (origin+h)%13==0 else 0)\n        inflow=np.minimum(proposed,np.maximum(a-current+out,0))\n        current=current+inflow-out\n        arr[:,h-1]=np.maximum(a-current,0)\n        flows.append((out.mean(),inflow.mean()))\n    return arr,dict(origin=origin,paths=n_paths,seed=seed,lam=lam,extra_out=extra_out,available_through=int(g.week.max()),flow_means=flows)\n\ndef forecast_table(df,origin=91,seed=0,lam=0.0):\n    rows=[];series=[];paths_by={}\n    for idx,(u,r) in enumerate([(u,r) for u in UNITS for r in ROLES]):\n        g=get_cell(df,u,r,origin);paths,ass=mc_paths(g,origin,seed=seed+idx,lam=lam);paths_by[(u,r)]=paths\n        peaks=paths.max(axis=1);a=int(g.authorized.iloc[-1]);now=derive(g).iloc[-1]\n        rows.append(dict(unit=u,role=r,origin=origin,authorized=a,current=int(now.current),vacancy=int(now.vacancy),available=int(now.available),overtime_h=float(now.overtime_h),night_ratio=float(now.night_ratio),p_need=float((peaks/a>=.15).mean()),peak_mean=float(peaks.mean()),need_median=int(np.ceil(np.median(peaks))),need_p90=int(np.ceil(np.percentile(peaks,90))),spread=float(np.percentile(peaks,90)-np.percentile(peaks,10)),data_ratio=float(g[g.week>origin-13].week.nunique()/13)))\n        for h in range(13):\n            q=np.percentile(paths[:,h],[10,50,90]);series.append(dict(unit=u,role=r,week=origin+h+1,origin=origin,q10=q[0],q50=q[1],q90=q[2]))\n    return pd.DataFrame(rows),pd.DataFrame(series),paths_by\n\ndef assess_risk(table,comparison=None):\n    t=table.copy();t[\'risk_prob\']=50*t.p_need;t[\'risk_peak\']=30*np.clip(t.peak_mean/t.authorized/.3,0,1);t[\'risk_load\']=20*np.clip(t.overtime_h/20,0,1)\n    t[\'risk\']=t[[\'risk_prob\',\'risk_peak\',\'risk_load\']].sum(axis=1)\n    t[\'agreement\']=1-np.minimum(np.abs(t.peak_mean-(comparison if comparison is not None else t.vacancy))/t.authorized/.2,1)\n    t[\'confidence_score\']=.4*(1-np.minimum(t.spread/t.authorized/.35,1))+.3*t.agreement+.3*t.data_ratio\n    t[\'confidence\']=np.select([t.confidence_score>=.7,t.confidence_score>=.45],[\'High\',\'Med\'],default=\'Low\')\n    t[\'grade\']=np.select([t.risk>=70,t.risk>=40],[\'높음\',\'주의\'],default=\'낮음\')\n    t.loc[t.overtime_h.isna(),[\'risk\',\'confidence_score\']]=np.nan\n    t.loc[t.overtime_h.isna(),\'confidence\']=\'Low\'\n    t.loc[t.overtime_h.isna(),\'grade\']=\'미확인\'\n    t[\'review_status\']=np.where(t.risk.isna()|((t.risk>=40)&(t.confidence==\'Low\')),\'근거 추가 확인\',\'인사관 검토 대기\')\n    return t\n\ndef payload(title,why,tables=None,metrics=None,note=\'\'):\n    return dict(title=title,why=why,tables=tables or {},metrics=native(metrics or {}),note=note)\n\nclass Audit:\n    def __init__(self):self.events=[]\n    def add(self,event,**body):\n        previous=self.events[-1][\'hash\'] if self.events else \'GENESIS\'\n        x=dict(seq=len(self.events),event=event,**native(body),previous=previous)\n        x[\'hash\']=hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest();self.events.append(x)\n        return x\n    def verify(self,anchor=None):\n        prev=\'GENESIS\'\n        for i,event in enumerate(self.events):\n            body={k:v for k,v in event.items() if k!=\'hash\'}\n            if event[\'seq\']!=i or event[\'previous\']!=prev or hashlib.sha256(json.dumps(body,sort_keys=True,ensure_ascii=False).encode()).hexdigest()!=event[\'hash\']:return False\n            prev=event[\'hash\']\n        return anchor is None or prev==anchor\n\nclass Workflow:\n    """모의 역할에 따라 허용되는 상태 전이를 검사하는 교육용 코드입니다."""\n    roles={\'clerk\':\'clerk\',\'reviewer\':\'reviewer\',\'commander\':\'commander\',\'executor\':\'executor\',\'system\':\'system\'}\n    permissions={\'submit\':\'clerk\',\'review\':\'reviewer\',\'adjust\':\'reviewer\',\'approve\':\'commander\',\'reject\':\'commander\',\'hold\':\'commander\',\'execute\':\'executor\',\'revise\':\'clerk\'}\n    def __init__(self,resources):\n        self.resources=resources.copy();self.cases={};self.audit=Audit();self.calls=0;self.executed=set()\n    def propose(self,rid,role,need,qty):\n        if rid in self.cases or role not in ROLES or isinstance(qty,bool) or not isinstance(qty,int) or qty<0 or qty>need:raise ValueError(\'잘못된 제안\')\n        self.cases[rid]=dict(request_id=rid,role=role,need=int(need),qty=int(qty),original=int(qty),version=1,state=\'DRAFT\',submitter=None,reviewer=None,reason=\'\')\n    def act(self,actor,rid,action,reason=\'\',qty=None,version=None,event_id=None):\n        self.calls+=1;c=self.cases.get(rid);before=copy.deepcopy(c)\n        def deny(why):self.audit.add(\'DENIED\',actor=actor,rid=rid,action=action,reason=why);return False,why\n        if actor not in self.roles or self.permissions.get(action)!=self.roles[actor]:return deny(\'권한 없음\')\n        if c is None:return deny(\'요청 없음\')\n        if version is not None and version!=c[\'version\']:return deny(\'검토한 버전과 다름\')\n        if not reason.strip():return deny(\'사유 누락\')\n        transitions={\'submit\':(\'DRAFT\',\'SUBMITTED\'),\'review\':(\'SUBMITTED\',\'REVIEWED\'),\'approve\':(\'REVIEWED\',\'APPROVED\'),\'reject\':(\'REVIEWED\',\'REJECTED\'),\'hold\':(\'REVIEWED\',\'HOLD\'),\'execute\':(\'APPROVED\',\'EXECUTED\')}\n        if action==\'adjust\':\n            if c[\'state\']!=\'REVIEWED\':return deny(\'검토 단계 아님\')\n            if isinstance(qty,bool) or not isinstance(qty,int) or qty<0 or qty>c[\'need\']:return deny(\'수량 범위 오류\')\n            if abs(qty-c[\'original\'])>max(1,math.ceil(.5*c[\'original\'])):return deny(\'조정 한도 초과\')\n            if qty>self.resources[c[\'role\']]:return deny(\'자원 부족\')\n            c[\'qty\']=qty;c[\'version\']+=1;c[\'state\']=\'SUBMITTED\';c[\'reviewer\']=None\n        elif action==\'revise\':\n            if c[\'state\'] not in [\'REJECTED\',\'HOLD\']:return deny(\'수정 가능한 상태 아님\')\n            c[\'version\']+=1;c[\'state\']=\'DRAFT\';c[\'reviewer\']=None\n        else:\n            expected,nxt=transitions[action]\n            if c[\'state\']!=expected:return deny(\'단계 순서 오류\')\n            if action==\'submit\':c[\'submitter\']=actor\n            if action==\'review\':\n                if actor==c[\'submitter\']:return deny(\'작성자와 검토자 분리\')\n                c[\'reviewer\']=actor\n            if action==\'approve\':\n                if actor in [c[\'submitter\'],c[\'reviewer\']]:return deny(\'승인자 분리\')\n                if c[\'qty\']>self.resources[c[\'role\']]:return deny(\'중복 예약 또는 자원 부족\')\n                self.resources[c[\'role\']]-=c[\'qty\']\n            if action==\'execute\':\n                if not event_id or event_id in self.executed:return deny(\'집행 이벤트 누락 또는 중복\')\n                self.executed.add(event_id)\n            c[\'state\']=nxt\n        c[\'reason\']=reason\n        self.audit.add(\'ACTION\',actor=actor,rid=rid,action=action,before=before,after=copy.deepcopy(c),reason=reason,event_id=event_id)\n        return True,c[\'state\']\n\nclass ForecastService:\n    def __init__(self,df,model_version=\'mc-2\'):\n        self.df=df.copy();self.data_version=fingerprint(df);self.model_version=model_version;self.cache={};self.fallback_cache={};self.lock=threading.Lock();self.heavy_calls=0\n    def key(self,unit,role,origin,lam):return (self.data_version,self.model_version,unit,role,int(origin),float(lam))\n    def invalidate(self,df=None,model_version=None):\n        if df is not None:self.df=df.copy();self.data_version=fingerprint(df)\n        if model_version is not None:self.model_version=model_version\n        self.cache.clear();self.fallback_cache.clear()\n    def predict(self,unit,role,origin=91,lam=0,optimized=True,fail=False):\n        if unit not in UNITS or role not in ROLES:raise KeyError(\'잘못된 조직 또는 직무\')\n        if fail:raise RuntimeError(\'주입한 모델 오류\')\n        k=self.key(unit,role,origin,lam)\n        if optimized:\n            with self.lock:\n                if k in self.cache:return dict(self.cache[k],cache_hit=True)\n        g=get_cell(self.df,unit,role,origin)\n        paths,_=mc_paths(g,origin,seed=0,lam=lam);self.heavy_calls+=1\n        ans=dict(origin=origin,data_through=int(g.week.max()),need=int(np.ceil(np.median(paths.max(axis=1)))),mode=\'NORMAL\',cache_hit=False)\n        if optimized:\n            with self.lock:self.cache[k]=ans.copy()\n        return ans\n    def fallback(self,unit,role,origin):\n        if unit not in UNITS or role not in ROLES:raise KeyError(\'잘못된 조직 또는 직무\')\n        key=self.key(unit,role,origin,0)\n        if key in self.fallback_cache:return self.fallback_cache[key].copy()\n        g=get_cell(self.df,unit,role,origin);v=int(max(g.authorized.iloc[-1]-g.current.iloc[-1],0))\n        ans=dict(origin=origin,data_through=int(g.week.max()),need=v,mode=\'DEGRADED\',method=\'현재 결원 유지 규칙\')\n        self.fallback_cache[key]=ans.copy();return ans\n\ndef stress_run(service,rate,duration=1,optimized=True,workers=4,qmax=32):\n    if rate<=0 or duration<=0:raise ValueError(\'도착률과 시험시간은 양수\')\n    q=queue.Queue(qmax);results=[];lock=threading.Lock();start=time.perf_counter();n=int(rate*duration)\n    def respond(item,full=False):\n        idx,u,r,origin,due,arrived=item;begin=time.perf_counter();error=\'\';ans=None\n        try:\n            if full:ans=service.fallback(u,r,origin);status=\'DEGRADED_QUEUE\'\n            else:\n                try:ans=service.predict(u,r,origin,optimized=optimized,fail=idx%97==7);status=\'NORMAL\'\n                except RuntimeError:ans=service.fallback(u,r,origin);status=\'DEGRADED_MODEL\'\n        except KeyError as exc:status=\'QUARANTINED\';error=str(exc)\n        except Exception as exc:status=\'FAILED\';error=repr(exc)\n        end=time.perf_counter()\n        with lock:results.append(dict(request_id=idx,scheduled=due-start,arrived=arrived-start,started=begin-start,ended=end-start,status=status,latency_ms=(end-arrived)*1000,queue_ms=(begin-arrived)*1000,schedule_lag_ms=(arrived-due)*1000,error=error,response=ans))\n    def worker():\n        while True:\n            item=q.get()\n            try:\n                if item is None:return\n                respond(item)\n            finally:q.task_done()\n    threads=[threading.Thread(target=worker,daemon=True) for _ in range(workers)]\n    for t in threads:t.start()\n    for idx in range(n):\n        due=start+idx/rate;left=due-time.perf_counter()\n        if left>0:time.sleep(left)\n        item=(idx,UNITS[idx%4],ROLES[idx%6] if idx%101 else \'J99\',78 if idx%2 else 91,due,time.perf_counter())\n        try:q.put_nowait(item)\n        except queue.Full:respond(item,full=True)\n    enqueue_end=time.perf_counter();q.join()\n    for _ in threads:q.put(None)\n    for t in threads:t.join()\n    trace=pd.DataFrame(results).sort_values(\'request_id\');elapsed=time.perf_counter()-start\n    counts=trace.status.value_counts().to_dict();lost=n-trace.request_id.nunique();duplicate=len(trace)-trace.request_id.nunique()\n    produced_rate=n/max(enqueue_end-start,1e-9)\n    summary=dict(target_rate=rate,duration=duration,optimized=optimized,generated=n,actual_rate=produced_rate,elapsed=elapsed,completed=len(trace),lost=int(lost),duplicates=int(duplicate),failed=int(counts.get(\'FAILED\',0)),p95_ms=float(trace.latency_ms.quantile(.95)),schedule_lag_p95_ms=float(trace.schedule_lag_ms.quantile(.95)),states=counts,full_duration=duration>=60,rate_met=produced_rate>=.95*rate)\n    return trace,summary\n\nCSS=\'\'\'*{box-sizing:border-box}body{margin:0;background:#f4f6f8;color:#182635;font:15px system-ui,sans-serif}main{display:grid;grid-template-columns:150px 1fr;min-height:100vh}nav{padding:20px 12px;background:#19354c;color:white}nav button{display:block;width:100%;margin:10px 0;padding:12px;border:1px solid #7e9aad;background:transparent;color:white;cursor:pointer;text-align:left}nav button.active{background:#f0f6fb;color:#19354c}article{padding:26px;min-width:0}table{border-collapse:collapse;width:100%;font-size:13px;background:white}td,th{border:1px solid #cdd8e0;padding:7px;text-align:left}section.step[hidden]{display:none}pre{white-space:pre-wrap;background:#e7edf2;padding:12px}input,select,button{padding:8px;margin:4px}h1{font-size:24px}h2{font-size:20px}.note{line-height:1.7}.night{background:#111c28;color:#e4edf7}.night table,.night pre{background:#203346;color:#e4edf7}svg{max-width:100%}.review{color:#a83514}.night .review{color:#ffc49b}@media(max-width:700px){main{grid-template-columns:105px 1fr}article{padding:12px}}\'\'\'\nJS=\'\'\'const nav=[...document.querySelectorAll(\'nav button\')];function choose(n){document.querySelectorAll(\'section.step\').forEach((s,i)=>s.hidden=i!==n);nav.forEach((b,i)=>b.classList.toggle(\'active\',i===n));}nav.forEach((b,i)=>b.onclick=()=>choose(i));choose(0);window.uiEvidence=null;\nfunction dashboardFilter(){let u=document.getElementById(\'unitFilter\')?.value||\'\';let r=document.getElementById(\'roleFilter\')?.value||\'\';document.querySelectorAll(\'#liveRows tr\').forEach(tr=>tr.hidden=(u&&tr.dataset.unit!==u)||(r&&tr.dataset.role!==r));}\nfunction drill(u,r){let x=window.dashboardData.find(x=>x.unit===u&&x.role===r);document.getElementById(\'detail\').textContent=JSON.stringify(x,null,2);let series=window.forecastSeries.filter(x=>x.unit===u&&x.role===r);let max=Math.max(1,...series.map(x=>x.q90));let pts=key=>series.map((x,i)=>`${30+i*38},${180-x[key]/max*145}`).join(\' \');document.getElementById(\'chart\').innerHTML=`<svg viewBox="0 0 530 210" aria-label="13주 결원 예측"><polyline points="${pts(\'q10\')}" fill="none" stroke="#789"/><polyline points="${pts(\'q90\')}" fill="none" stroke="#789"/><polyline points="${pts(\'q50\')}" fill="none" stroke="#d47632" stroke-dasharray="5 4"/><text x="20" y="205" fill="currentColor">점선: 예측 중앙값 / 위아래 선: 10·90백분위 / 단위 명</text></svg>`;}\nasync function nextPaint(){await new Promise(r=>requestAnimationFrame(()=>requestAnimationFrame(r)));}\nasync function measureUi(){let times={filter:[],drill:[],night:[]};const before=JSON.stringify(window.dashboardData);for(let i=0;i<50;i++){let t=performance.now();document.getElementById(\'unitFilter\').value=i%2?\'B01\':\'\';dashboardFilter();await nextPaint();times.filter.push(performance.now()-t);t=performance.now();drill(\'B01\',\'J01\');await nextPaint();times.drill.push(performance.now()-t);t=performance.now();document.body.classList.toggle(\'night\');await nextPaint();times.night.push(performance.now()-t);}let q=x=>[...x].sort((a,b)=>a-b)[Math.ceil(.95*x.length)-1];window.uiEvidence={scope:\'사용자 브라우저 두 번의 화면 갱신까지\',runs:50,p95_ms:Object.fromEntries(Object.entries(times).map(([k,v])=>[k,q(v)])),raw:times,values_unchanged:before===JSON.stringify(window.dashboardData),userAgent:navigator.userAgent};document.getElementById(\'uiTiming\').textContent=JSON.stringify(window.uiEvidence,null,2);}\nfunction saveEvidence(){let b=new Blob([JSON.stringify({mission:document.title,uiEvidence:window.uiEvidence,notes:document.getElementById(\'learnerNotes\')?.value||\'\',evidence:window.pythonEvidence},null,2)],{type:\'application/json\'});let a=document.createElement(\'a\');a.href=URL.createObjectURL(b);a.download=document.title+\'_record.json\';a.click();URL.revokeObjectURL(a.href);}\n\'\'\'\n\ndef dashboard_html(table,series):\n    cols=[\'unit\',\'role\',\'current\',\'vacancy\',\'available\',\'risk\',\'confidence\',\'need_median\',\'review_status\']\n    rows=\'\'.join(\'<tr data-unit="\'+html.escape(r[\'unit\'])+\'" data-role="\'+html.escape(r[\'role\'])+\'">\'+\'\'.join(\'<td>\'+html.escape(str(round(r[c],2) if isinstance(r[c],float) else r[c]))+\'</td>\' for c in cols)+f\'<td><button onclick="drill(\\\'{r["unit"]}\\\',\\\'{r["role"]}\\\')">상세 보기</button></td></tr>\' for r in table.to_dict(\'records\'))\n    lookup={(r[\'unit\'],r[\'role\']):r for r in table.to_dict(\'records\')}\n    heat=\'<h3>전체 24개 조합의 위험 점수</h3><p>점수와 상태를 함께 읽으세요. 자료 공백은 미확인으로 표시합니다.</p><table><thead><tr><th>조직</th>\'+\'\'.join(\'<th>\'+r+\'</th>\' for r in ROLES)+\'</tr></thead><tbody>\'\n    for unit in UNITS:\n        heat+=\'<tr><th>\'+unit+\'</th>\'\n        for role in ROLES:\n            r=lookup[(unit,role)];value=r[\'risk\'];color=\'#e7edf2\' if pd.isna(value) else \'#ffd8cf\' if value>=70 else \'#ffedbe\' if value>=40 else \'#e4f0e9\'\n            label=\'미확인\' if pd.isna(value) else str(round(value,1))+\'점 \'+str(r[\'grade\'])\n            heat+=\'<td style="background:\'+color+\';color:#182635">\'+label+\'</td>\'\n        heat+=\'</tr>\'\n    heat+=\'</tbody></table><h3>조직과 직무를 골라 상세 확인</h3>\'\n    data=json.dumps(native(table),ensure_ascii=False).replace(\'<\',\'\\\\u003c\');forecast=json.dumps(native(series),ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    return \'<div id="dashboard">\'+heat+\'<p>이 화면의 수치는 교육용 자료입니다. 제안 인원은 승인이나 실제 충원 결과가 아닙니다.</p><label>조직<select id="unitFilter" onchange="dashboardFilter()"><option value="">전체</option>\'+\'\'.join(\'<option>\'+x+\'</option>\' for x in UNITS)+\'</select></label><label>직무<select id="roleFilter" onchange="dashboardFilter()"><option value="">전체</option>\'+\'\'.join(\'<option>\'+x+\'</option>\' for x in ROLES)+\'</select></label><button onclick="document.body.classList.toggle(\\\'night\\\')">야간 모드</button><table><thead><tr>\'+\'\'.join(\'<th>\'+x+\'</th>\' for x in [\'조직\',\'직무\',\'현재원\',\'편제상 결원\',\'근무 가용\',\'위험 점수\',\'근거 신뢰도\',\'제안 소요\',\'검토 상태\'])+\'<th>상세</th></tr></thead><tbody id="liveRows">\'+rows+\'</tbody></table><pre id="detail">상세 보기를 누르세요.</pre><div id="chart"></div><button onclick="measureUi()">실제 화면 응답 50회 측정</button><pre id="uiTiming">브라우저에서 측정하지 않았습니다.</pre></div><script>window.dashboardData=\'+data+\';window.forecastSeries=\'+forecast+\';</script>\'\n\ndef export_report(root,mid,stages):\n    if len(stages)!=5 or any(not isinstance(s,dict) or not s.get(\'title\') for s in stages):raise ValueError(\'실제 단계 결과 5개 필요\')\n    root=Path(root);out=root/\'outputs\'/mid;out.mkdir(parents=True,exist_ok=True);sections=[]\n    for i,s in enumerate(stages,1):\n        blocks=[\'<h2>STEP %02d %s</h2>\'%(i,html.escape(s[\'title\'])),\'<p class="note">\'+html.escape(s[\'why\'])+\'</p>\',\'<pre>\'+html.escape(json.dumps(native(s[\'metrics\']),ensure_ascii=False,indent=2))+\'</pre>\']\n        for name,tab in s[\'tables\'].items():\n            if not isinstance(tab,pd.DataFrame):raise TypeError(\'결과 표는 DataFrame이어야 함\')\n            tab.to_csv(out/f\'STEP{i:02}_{re.sub(r"[^A-Za-z0-9_]","_",name)}.csv\',index=False,encoding=\'utf-8-sig\')\n            blocks+=[\'<h3>\'+html.escape(name)+\'</h3>\',tab.head(100).to_html(index=False,escape=True),\'<p>화면에는 최대 100행을 표시하며 전체 자료는 같은 폴더의 CSV에 저장했습니다.</p>\']\n        blocks+=[\'<p class="note">\'+html.escape(s.get(\'note\',\'\'))+\'</p>\',s.get(\'html\',\'\')]\n        sections.append(\'<section class="step"\'+(\' hidden\' if i>1 else \'\')+\'>\'+\'\'.join(blocks)+\'</section>\')\n    evidence=dict(version=VERSION,mission=mid,steps=[{k:native(v) for k,v in s.items() if k not in [\'tables\',\'html\']} for s in stages],automatic_pass=False)\n    safe=json.dumps(evidence,ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    doc=\'<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>ADM01_\'+mid+\'</title><style>\'+CSS+\'</style></head><body><main><nav><strong>\'+mid+\'</strong>\'+\'\'.join(\'<button>STEP %02d</button>\'%i for i in range(1,6))+\'</nav><article><h1>\'+mid+\' 실제 코드 실행 결과</h1>\'+\'\'.join(sections)+\'<h2>결과를 읽고 남길 설명</h2><p>가장 중요한 결과, 그 근거가 된 표와 단위, 아직 확인하지 못한 사항, 다음 담당자가 해야 할 일을 적으세요.</p><textarea id="learnerNotes" rows="6" style="width:100%"></textarea><button onclick="saveEvidence()">판단 기록 JSON 저장</button><script>window.pythonEvidence=\'+safe+\';</script><script>\'+JS+\'</script></article></main></body></html>\'\n    (out/f\'ADM01_{mid}_result.html\').write_text(doc,encoding=\'utf-8\');dump(out/f\'ADM01_{mid}_code_report.json\',evidence)\n    return out/f\'ADM01_{mid}_result.html\'\n\ndef init_state(root=\'ADM01_work\',seed=7,stress_seconds=1):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    for sub in [\'data\',\'docs\',\'outputs\',\'configs\',\'verification\']:(root/sub).mkdir(exist_ok=True)\n    weekly,labels=make_weekly(seed);raw,gold=make_logs()\n    return dict(root=root,weekly=weekly,load_labels=labels,raw_logs=raw,log_truth=gold,seed=seed,stress_seconds=stress_seconds,cache={})\n'
runtime_file = PROJECT_HOME / 'adm01_runtime.py'
runtime_file.write_text(COMMON_CODE, encoding='utf-8')
sys.path.insert(0, str(PROJECT_HOME))
# 같은 커널에서 준비 셀을 다시 실행해도 최신 공통 코드를 불러옵니다.
sys.modules.pop('adm01_runtime', None)
from adm01_runtime import *
import importlib.util, platform
def forecast_bundle(S,origin=91,seed=0,lam=0):
    key=('forecast',origin,seed,lam)
    if key not in S['cache']:S['cache'][key]=forecast_table(S['weekly'],origin,seed,lam)
    return S['cache'][key]

def risk_bundle(S):
    if 'risk' not in S['cache']:
        t,series,paths=forecast_bundle(S);model=fit_ridge(S['weekly'],91)
        comparison=np.array([ridge_predict(model,get_cell(S['weekly'],r.unit,r.role,91),91).max() for r in t.itertuples()])
        S['cache']['risk']=(assess_risk(t,comparison),series,paths)
    return S['cache']['risk']

def workflow_demo(resources=None):
    w=Workflow(resources or {r:8 for r in ROLES});w.propose('Q1','J01',12,5)
    assert w.act('clerk','Q1','submit','검토 요청')[0];assert w.act('reviewer','Q1','review','수량 확인')[0]
    return w

def violation_tests():
    rows=[]
    def check(name,fn):
        try:result=fn();denied=not result[0];detail=result[1]
        except Exception as exc:denied=False;detail=repr(exc)
        rows.append(dict(test=name,blocked=denied,detail=detail))
    check('01 작성자의 승인',lambda:workflow_demo().act('clerk','Q1','approve','승인 시도'))
    check('02 시스템 자동 승인',lambda:workflow_demo().act('system','Q1','approve','승인 시도'))
    def before_review():
        w=Workflow({'J01':8});w.propose('Q1','J01',12,5);return w.act('commander','Q1','approve','검토 생략')
    check('03 검토 순서 생략',before_review)
    def same_actor():
        w=workflow_demo();w.roles=w.roles.copy();w.roles['clerk']='commander';return w.act('clerk','Q1','approve','역할 우회')
    check('04 같은 행위자 역할 우회',same_actor)
    check('05 음수 수량',lambda:workflow_demo().act('reviewer','Q1','adjust','수량 변경',qty=-1))
    check('06 소요 초과',lambda:workflow_demo().act('reviewer','Q1','adjust','수량 변경',qty=13))
    check('07 자원 초과',lambda:workflow_demo({'J01':1}).act('reviewer','Q1','adjust','수량 변경',qty=6))
    def duplicate_resource():
        w=workflow_demo({'J01':5});w.act('commander','Q1','approve','승인');w.propose('Q2','J01',5,5);w.act('clerk','Q2','submit','상신');w.act('reviewer','Q2','review','검토');return w.act('commander','Q2','approve','중복 예약')
    check('08 자원 중복 예약',duplicate_resource)
    check('09 조정 한도 초과',lambda:workflow_demo({'J01':20}).act('reviewer','Q1','adjust','조정',qty=10))
    check('10 조정 사유 누락',lambda:workflow_demo().act('reviewer','Q1','adjust','',qty=4))
    check('11 반려 사유 누락',lambda:workflow_demo().act('commander','Q1','reject',''))
    check('12 오래된 버전',lambda:workflow_demo().act('commander','Q1','approve','승인',version=999))
    check('13 미승인 집행',lambda:workflow_demo().act('executor','Q1','execute','집행',event_id='E1'))
    def replay():
        w=workflow_demo({'J01':12});w.act('commander','Q1','approve','승인');w.act('executor','Q1','execute','집행',event_id='E1')
        w.propose('Q2','J01',5,5);w.act('clerk','Q2','submit','상신');w.act('reviewer','Q2','review','검토');w.act('commander','Q2','approve','승인');return w.act('executor','Q2','execute','중복 이벤트',event_id='E1')
    check('14 집행 이벤트 중복',replay)
    return pd.DataFrame(rows)

def queue_times(priority=False):
    items=[dict(id=i,arrival=float(i),risk=[20,80,30,90,50,70,10,85,40,60][i]) for i in range(10)];pending=items.copy();now=0;rows=[]
    while pending:
        ready=[r for r in pending if r['arrival']<=now]
        if not ready:now=min(r['arrival'] for r in pending);continue
        chosen=max(ready,key=lambda r:r['risk']+3*(now-r['arrival'])) if priority else min(ready,key=lambda r:r['arrival'])
        rows.append(dict(**chosen,start=now,wait=now-chosen['arrival']));pending.remove(chosen);now+=2
    return pd.DataFrame(rows)

def evaluate_worlds(worlds,lam,p0):
    rows=[]
    for seed,df in worlds:
        table,_,_=forecast_table(df,91,seed=0,lam=lam)
        for r in table.itertuples():
            future=derive(df[(df.unit==r.unit)&(df.role==r.role)&(df.week>91)])
            truth=int(future.vacancy.max());actual_flag=bool((future.vacancy_rate>=.15).any())
            rows.append(dict(seed=seed,unit=r.unit,role=r.role,shock_group=r.unit in ['B01','B02'],predicted=int(r.need_median),actual=truth,error=int(r.need_median)-truth,p_need=r.p_need,actual_need=actual_flag,predicted_need=bool(r.p_need>=p0)))
    return pd.DataFrame(rows)

def world_metrics(df):
    return dict(within_two_accuracy=float(df.error.abs().le(2).mean()),mae=float(df.error.abs().mean()),bias=float(df.error.mean()),**binary_metrics(df.actual_need,df.predicted_need))

def pipeline(df,logs=None,use_mc=True):
    clean,quarantine=validate(df);log_results=preprocess(logs) if logs is not None else pd.DataFrame();audit=Audit();audit.add('LOAD',rows=len(df),quarantined=len(quarantine))
    origin=int(clean.week.max());now=derive(clean[clean.week==origin]);now['load_score']=load_score(now)
    if use_mc:
        try:table,series,_=forecast_table(clean,origin);result=assess_risk(table);mode='MODEL'
        except (ValueError,RuntimeError) as exc:use_mc=False;audit.add('MODEL_FALLBACK',reason=str(exc))
    if not use_mc:
        result=now[['unit','role','authorized','current','vacancy','available','overtime_h']].copy();result['need_median']=result.vacancy.astype(int);result['risk']=np.clip(result.vacancy/result.authorized/.3,0,1)*100;result['confidence']='Low';result['review_status']='예측 불가로 현재 결원 확인';series=pd.DataFrame();mode='RULE'
    queue_=result.copy();queue_['state']='DRAFT'
    for r in queue_.itertuples():audit.add('PROPOSAL',unit=r.unit,role=r.role,need=int(r.need_median),state='DRAFT')
    return dict(clean=clean,quarantine=quarantine,logs=log_results,current=now,result=result,series=series,queue=queue_,audit=audit,mode=mode)
STATE = init_state(PROJECT_HOME, seed=7, stress_seconds=1)
print('준비 완료: 주간 인력 자료', len(STATE['weekly']), '행 / 근무일지', len(STATE['raw_logs']), '줄')
print('M8 기본 실행은 조건별 1초 기능 검사입니다. 60초 부하 시험은 패키지의 run_stress.py로 별도 실행하세요.')


## M08 전출 증가와 조회 급증을 나누어 시험하기

### 상황

B02/J04의 가상 전출이 증가한 날 조회 요청도 늘었다. 인원 부족이 커진 이유와 화면 응답이 느려진 이유는 서로 다르다. 캐시가 옛 주차의 결과를 최신 결과처럼 내놓지 않는지도 확인해야 한다.

### 이해할 개념

부하 시험은 실제로 많은 요청을 발생시켜 처리시간과 실패 상태를 확인하는 시험이다. 캐시는 같은 계산의 이전 결과를 재사용하는 방법이다. 오류가 나서 현재 결원 규칙으로 대신 응답했다면 원래 예측과 구분해 표시해야 한다.

### 해결할 일

전출 가정의 변화와 조회 증가를 나누어 비교한다. 150·300·750건/초를 최적화 전후에 시험하며, 짧은 기능 검사는 1초, 정식 부하 시험은 조건별 60초로 구분한다.

미션을 처음부터 다시 실행하려면 아래 준비 셀부터 시작하세요. 원자료는 공통 준비 셀에서 생성한 자료를 사용합니다.


In [ ]:
MISSION_STAGES = []
MID = 'M08'


### STEP 01 조건 분리

**왜 이 단계가 필요한가**

기준 150건/초의 500% 부하는 750건/초다. 500% 증가라면 900건/초이므로 같은 뜻으로 쓰지 않는다.

**코드로 작성할 내용**

기본 자료, 시드, 예측시점을 고정하고 추가 전출과 조회 도착률을 독립 조건으로 둔다.

함수는 `STATE`를 받아 표·지표·설명이 담긴 `payload(...)`를 반환합니다. 이후 단계에서 필요한 중간값은 `STATE`에 저장합니다. 영어로 된 열 이름은 자료 항목 식별자이며 값의 뜻과 단위는 스키마(`default_schema`)와 과제 안내에 정리했습니다.


In [ ]:
def m08_s1(S):
    """기본 자료, 시드, 예측시점을 고정하고 추가 전출과 조회 도착률을 독립 조건으로 둔다."""
    # 이 함수에서 계산 결과를 만들고 필요한 중간값을 S에 저장하세요.
    # 입력과 계산 근거는 바로 위 설명을 따릅니다.
    raise NotImplementedError("M08 STEP 01: 조건 분리 함수를 작성하세요.")

# 같은 단계를 다시 실행하면 그 단계 이후의 이전 출력은 제거합니다.
MISSION_STAGES = MISSION_STAGES[:0]
if len(MISSION_STAGES) != 0:
    raise RuntimeError('이 미션의 이전 단계를 먼저 실행하세요.')
result = m08_s1(STATE)
MISSION_STAGES.append(result)
print(result['title'])
print(json.dumps(native(result['metrics']), ensure_ascii=False, indent=2))
for table_name, table in result['tables'].items():
    print(table_name, '전체', len(table), '행')
    print(table.head(8).to_string(index=False))


### STEP 02 전출 증가 영향

**왜 이 단계가 필요한가**

인원 감소에서 초과근무를 자동으로 확정하거나 개인 피로를 진단하지 않는다. 근무시간 조정에는 별도 자료와 사람의 검토가 필요하다.

**코드로 작성할 내용**

B02/J04의 향후 유출 가정에 주당 평균 1명을 추가하고 다른 직무와 함께 13주 보충량 변화를 비교한다.

함수는 `STATE`를 받아 표·지표·설명이 담긴 `payload(...)`를 반환합니다. 이후 단계에서 필요한 중간값은 `STATE`에 저장합니다. 영어로 된 열 이름은 자료 항목 식별자이며 값의 뜻과 단위는 스키마(`default_schema`)와 과제 안내에 정리했습니다.


In [ ]:
def m08_s2(S):
    """B02/J04의 향후 유출 가정에 주당 평균 1명을 추가하고 다른 직무와 함께 13주 보충량 변화를 비교한다."""
    # 이 함수에서 계산 결과를 만들고 필요한 중간값을 S에 저장하세요.
    # 입력과 계산 근거는 바로 위 설명을 따릅니다.
    raise NotImplementedError("M08 STEP 02: 전출 증가 영향 함수를 작성하세요.")

# 같은 단계를 다시 실행하면 그 단계 이후의 이전 출력은 제거합니다.
MISSION_STAGES = MISSION_STAGES[:1]
if len(MISSION_STAGES) != 1:
    raise RuntimeError('이 미션의 이전 단계를 먼저 실행하세요.')
result = m08_s2(STATE)
MISSION_STAGES.append(result)
print(result['title'])
print(json.dumps(native(result['metrics']), ensure_ascii=False, indent=2))
for table_name, table in result['tables'].items():
    print(table_name, '전체', len(table), '행')
    print(table.head(8).to_string(index=False))


### STEP 03 실제 요청 발생

**왜 이 단계가 필요한가**

요청 발생기가 목표 속도를 내지 못할 수도 있다. 예정 도착률과 실제 도착률, 도착 지연을 따로 보고한다.

**코드로 작성할 내용**

예측 함수를 실제로 호출하며 요청 ID, 예정 도착, 실제 도착, 시작, 종료, 처리 상태를 기록한다.

함수는 `STATE`를 받아 표·지표·설명이 담긴 `payload(...)`를 반환합니다. 이후 단계에서 필요한 중간값은 `STATE`에 저장합니다. 영어로 된 열 이름은 자료 항목 식별자이며 값의 뜻과 단위는 스키마(`default_schema`)와 과제 안내에 정리했습니다.


In [ ]:
def m08_s3(S):
    """예측 함수를 실제로 호출하며 요청 ID, 예정 도착, 실제 도착, 시작, 종료, 처리 상태를 기록한다."""
    # 이 함수에서 계산 결과를 만들고 필요한 중간값을 S에 저장하세요.
    # 입력과 계산 근거는 바로 위 설명을 따릅니다.
    raise NotImplementedError("M08 STEP 03: 실제 요청 발생 함수를 작성하세요.")

# 같은 단계를 다시 실행하면 그 단계 이후의 이전 출력은 제거합니다.
MISSION_STAGES = MISSION_STAGES[:2]
if len(MISSION_STAGES) != 2:
    raise RuntimeError('이 미션의 이전 단계를 먼저 실행하세요.')
result = m08_s3(STATE)
MISSION_STAGES.append(result)
print(result['title'])
print(json.dumps(native(result['metrics']), ensure_ascii=False, indent=2))
for table_name, table in result['tables'].items():
    print(table_name, '전체', len(table), '행')
    print(table.head(8).to_string(index=False))


### STEP 04 캐시와 오류 대체

**왜 이 단계가 필요한가**

78주 요청에 104주 자료를 쓰면 미래 정보가 섞인다. 빠르다는 이유로 시점이 틀린 응답을 성공으로 보면 안 된다.

**코드로 작성할 내용**

자료 지문, 모델 버전, 조직, 직무, 예측시점, 가정을 캐시 키에 넣는다. 오류 대체 결과에도 사용한 시점을 표시한다.

함수는 `STATE`를 받아 표·지표·설명이 담긴 `payload(...)`를 반환합니다. 이후 단계에서 필요한 중간값은 `STATE`에 저장합니다. 영어로 된 열 이름은 자료 항목 식별자이며 값의 뜻과 단위는 스키마(`default_schema`)와 과제 안내에 정리했습니다.


In [ ]:
def m08_s4(S):
    """자료 지문, 모델 버전, 조직, 직무, 예측시점, 가정을 캐시 키에 넣는다. 오류 대체 결과에도 사용한 시점을 표시한다."""
    # 이 함수에서 계산 결과를 만들고 필요한 중간값을 S에 저장하세요.
    # 입력과 계산 근거는 바로 위 설명을 따릅니다.
    raise NotImplementedError("M08 STEP 04: 캐시와 오류 대체 함수를 작성하세요.")

# 같은 단계를 다시 실행하면 그 단계 이후의 이전 출력은 제거합니다.
MISSION_STAGES = MISSION_STAGES[:3]
if len(MISSION_STAGES) != 3:
    raise RuntimeError('이 미션의 이전 단계를 먼저 실행하세요.')
result = m08_s4(STATE)
MISSION_STAGES.append(result)
print(result['title'])
print(json.dumps(native(result['metrics']), ensure_ascii=False, indent=2))
for table_name, table in result['tables'].items():
    print(table_name, '전체', len(table), '행')
    print(table.head(8).to_string(index=False))


### STEP 05 요청 보존과 판정

**왜 이 단계가 필요한가**

잘못된 요청을 격리했다는 사실과 정상 업무 응답에 성공했다는 사실은 다르다. 누락된 요청 ID가 없어야 유실 0건을 말할 수 있다.

**코드로 작성할 내용**

입력 요청수와 정상, 대체, 격리, 실패 응답수를 맞추고 유실·중복을 검사한다. 짧은 검사와 60초 시험의 판정을 분리한다.

함수는 `STATE`를 받아 표·지표·설명이 담긴 `payload(...)`를 반환합니다. 이후 단계에서 필요한 중간값은 `STATE`에 저장합니다. 영어로 된 열 이름은 자료 항목 식별자이며 값의 뜻과 단위는 스키마(`default_schema`)와 과제 안내에 정리했습니다.


In [ ]:
def m08_s5(S):
    """입력 요청수와 정상, 대체, 격리, 실패 응답수를 맞추고 유실·중복을 검사한다. 짧은 검사와 60초 시험의 판정을 분리한다."""
    # 이 함수에서 계산 결과를 만들고 필요한 중간값을 S에 저장하세요.
    # 입력과 계산 근거는 바로 위 설명을 따릅니다.
    raise NotImplementedError("M08 STEP 05: 요청 보존과 판정 함수를 작성하세요.")

# 같은 단계를 다시 실행하면 그 단계 이후의 이전 출력은 제거합니다.
MISSION_STAGES = MISSION_STAGES[:4]
if len(MISSION_STAGES) != 4:
    raise RuntimeError('이 미션의 이전 단계를 먼저 실행하세요.')
result = m08_s5(STATE)
MISSION_STAGES.append(result)
print(result['title'])
print(json.dumps(native(result['metrics']), ensure_ascii=False, indent=2))
for table_name, table in result['tables'].items():
    print(table_name, '전체', len(table), '행')
    print(table.head(8).to_string(index=False))
report_path = export_report(STATE['root'], MID, MISSION_STAGES)
print('미션 HTML:', report_path)
try:
    from IPython.display import display, HTML
    document = report_path.read_text(encoding='utf-8')
    display(HTML('<iframe title="미션 단계별 결과" sandbox="allow-scripts allow-downloads" style="width:100%;height:1050px;border:1px solid #ccd" srcdoc="'+html.escape(document, quote=True)+'"></iframe>'))
except ImportError:
    print('노트북 표시 도구가 없습니다. 생성된 HTML을 브라우저에서 여세요.')


### 교관이 확인할 증거

**인력 충격과 조회 부하를 나누어 시험했는가?**

같은 초기 자료의 전출 전후 비교와 150·300·750건/초의 실제 도착률을 제출한다. 1초 기능 검사와 조건별 60초 시험을 구분한다.

**오류 대체와 캐시 및 요청 보존이 검증되는가?**

주차별 캐시 분리와 무효화, 모델 오류·큐 포화 대체 표시, 입력 ID 전체 회계와 유실·중복·실패를 보고한다. 정식 5배 부하에서 다운 0건·유실 0%가 목표다.

기술 목표에 미달했다면 실제값과 이유, 채택 보류·추가 검증을 기록하세요. 수량 계산 오류나 실패 은폐는 허용하지 않습니다.
